[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YangKCLab/social-media-analysis/blob/main/docs/topics/data-collection/4chan_archive.ipynb)

# 4chan API: archive

List the threads that have expired on a board with `archive.json`. A thread expires when it is pushed off the board's last page. It is then read-only, stays available for a while, and is deleted. The archive is a plain list of thread IDs, most recent last.

The 4chan API is read-only JSON. It needs no account and no key. The only
dependency is `requests`, which Google Colab has preinstalled.

Rules from the [API documentation](https://github.com/4chan/4chan-API):
at most one request per second, poll a thread no more often than every 10
seconds, and disclose 4chan as the source of anything you publish from it.
Boards can contain offensive and not-safe-for-work content. Read the
"Research considerations" section on the topic page before collecting.

In [1]:
import requests

In [2]:
api_url = "https://a.4cdn.org/{board}/archive.json"
resp = requests.get(api_url.format(board="g"))
archived = resp.json()
len(archived)

1462

In [3]:
archived[:5], archived[-5:]

([109444547, 109495988, 109497359, 109504557, 109521167],
 [109677736, 109677783, 109678036, 109678268, 109678926])

Not every board has an archive. `boards.json` reports it in `is_archived`; at the time of writing `/b/`, `/bant/`, `/f/`, and `/trash/` have none. An archived thread is still readable through the thread endpoint until it is deleted, and its OP carries `archived` and `archived_on`.

In [4]:
thread_url = "https://a.4cdn.org/{board}/thread/{op_id}.json"
resp = requests.get(thread_url.format(board="g", op_id=archived[-1]))
op = resp.json()["posts"][0]
{field: op.get(field) for field in ["no", "time", "replies", "images", "archived", "archived_on"]}

{'no': 109678926,
 'time': 1788012460,
 'replies': 3,
 'images': 0,
 'archived': 1,
 'archived_on': 1788036212}

Why this matters for collection: an archived thread does not change any more, so fetching it once gives you the complete thread. Polling the archive, and fetching every ID you have not seen before, is the simplest way to collect a whole board. The next notebook does exactly that.